from 2022

# 工作流程

## 基本理论

通过一种特殊的提示工程来引导模型

每一步的输出分为
- reason
- action
- observation

不断重复 reason -> action -> observation , 直到触发预定义的终止条件(thought中内容符合要求), 公式化的表达如下: 

$$
(t h_t, a_t) = \pi \bigl(q, (a_1, o_1), \dots, (a_{t-1}, o_{t-1})\bigr)
$$

$$
o_t = T(a_t)
$$

**适用场景**
- 需要补充外部信息来完成决策(知识, 静态)
- 需要与外部完成信息交互(交互, 动态)
- 需要完成精确的计算任务(工具调用)

## action的方式-tool
三要素
- 名称
- 描述: 让llm理解这个工具
- 实体: 可以执行的函数或者方法

## 特点
- 武装llm, 格外重视工具调用
- 可解释性强
- 步步为营, 稳扎稳打

## 局限性
- 缺乏全局规划
- 高度依赖底层模型和prompt
- 串行调用, 执行效率低

In [1]:
%cd ../
import os
print(os.getcwd())

/Users/yqz/project/hello-agents
/Users/yqz/project/hello-agents


### 协议设计

llm_output格式: 参考上一章节的; 
```txt
{
  "thought": "...",
  "todo": {
    "name": "<what to do>",
    "args": { <param_name>: <param_value> }
  }
}
```
比如
终止条件为
```txt
{
  "thought": "...",
  "todo": {
    "name": "finish",
    "args": { "<param_name>": "<param_value>"}
  }
}
```
某个函数调用为, 它的description为"search(query: str): A web search engine tool based on SerpApi. It intelligently parses search results and prioritizes returning direct answers or knowledge graph information."
```txt
{
  "thought": "...",
  "todo": {
    "name": "search",
    "args": { "query": "Who is the current President of the United States?"  }
  }
}
```

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# ReActAgent类
import json
class ReActAgent:
    def __init__(self, llm, tool, max_step):
        self.client = llm
        self.tool = tool
        self.max_step = max_step
        
        self.history = []

    def run(self, prompt, system_prompt):
        self.history = []
        current_step = 0
        self.history.append(
            json.dumps({"role": "user", "content": prompt}, ensure_ascii=False)
        )
        while current_step < self.max_step:
            print('='*20 + f"第{current_step}轮" + '='*20)
            
            # print(f'input: \n{self.history}\n')
            full_prompt = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": "\n".join(self.history)}
            ]
            output = self.client.generate(
                messages=full_prompt
            )
            # print(f'output: \n{output}\n')

            result = json.loads(output)
            '''
            复习一下协议
            {
                "thought": "...",
                "todo": {
                    "name": "<what to do>",
                    "args": { <param_name>: <param_value> }
                }
            }
            '''
            if result.get("todo").get("name") == "finish":
                print(f'\n {result["todo"]["args"]} \n')
                print("finished")
                break
            else:
                args = result.get("todo").get("args")
                tool_function = self.tool.useTool(
                    result.get("todo").get("name")
                )
                tool_result = tool_function(**args)
                print(f'tool_result: \n{tool_result}\n')
                self.history.append(
                    json.dumps({"role": "tool", "content": tool_result}, ensure_ascii=False)
                )
            current_step += 1
        if current_step >= self.max_step:
            print("reach max step")

### 提示词构建
提供两个可供替代的内容 `question` 和 `introduceTool`

PS: 刚刚遇到个问题
json自带的 {}, 影响了 string.format的使用, 这里我改用 Template 模块解决

In [4]:
from string import Template
REACT_PROMPT = Template("""
You are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.

INPUT
- User question:
$question

- Available tools (you may ONLY use tools listed here; includes name, args, and description):
$introduceTool

OUTPUT FORMAT (must follow EXACTLY)
Return ONLY one valid JSON object:

{
  "thought": "...",
  "todo": {
    "name": "<tool_name_or_finish>",
    "args": { ... }
  }
}

RULES
1) Output ONLY JSON. No markdown, no extra text.
2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include detailed step-by-step reasoning.
3) If you choose a tool:
   - todo.name MUST be exactly one tool name from {introuceTool}.
   - todo.args MUST match that tool’s required parameters (correct names and types). Do not add extra params.
4) If you finish:
   - todo.name MUST be "finish"
   - todo.args MUST be: { "answer": "<final response to the user question>" }
5) Do NOT invent facts or tool results. If the question needs up-to-date or external info, call the most relevant tool first.
"""
)
REACT_PROMPT

In [5]:
# 应用提示词模版的示例
print(REACT_PROMPT.substitute(question="111", introduceTool="222"))


You are a Tool-Calling Agent. Decide the single next action to solve the user question using ONLY the tools listed below, or finish with a final answer.

INPUT
- User question:
111

- Available tools (you may ONLY use tools listed here; includes name, args, and description):
222

OUTPUT FORMAT (must follow EXACTLY)
Return ONLY one valid JSON object:

{
  "thought": "...",
  "todo": {
    "name": "<tool_name_or_finish>",
    "args": { ... }
  }
}

RULES
1) Output ONLY JSON. No markdown, no extra text.
2) "thought" must be a short, high-level plan (1–2 sentences). Do NOT include detailed step-by-step reasoning.
3) If you choose a tool:
   - todo.name MUST be exactly one tool name from {introuceTool}.
   - todo.args MUST match that tool’s required parameters (correct names and types). Do not add extra params.
4) If you finish:
   - todo.name MUST be "finish"
   - todo.args MUST be: { "answer": "<final response to the user question>" }
5) Do NOT invent facts or tool results. If the questi

### 整理工具

In [ ]:
from mylib import ToolExecutor, SEARCH_DESCRIPTION, search

# 初始化工具管理器
my_tool = ToolExecutor()

# 新建search工具(一个函数)
mysearch = search

# 加入到工具管理器
my_tool.registerTool(
    name="search",
    description=Tool.SEARCH_DESCRIPTION,
    function=mysearch
)

# 生成工具管理器初始化描述
introduceTool = my_tool.introduceTool()
print(introduceTool)

### 开始测试

In [ ]:
# 整合输入
input = "tell me the latest price of gold"

In [ ]:
system_prompt = REACT_PROMPT.substitute(
    introduceTool = introduceTool,
    question = input
)
print(system_prompt)

In [ ]:
my_llm = MyLLM.MyLLM()
agent = ReActAgent(
    llm=my_llm,
    tool=my_tool,
    max_step=10
)
agent.run(
    prompt=input,
    system_prompt=system_prompt
)

In [ ]:
from mylib.Tool import ToolExecutor
from mylib.Tool import search

In [ ]:
mytool = ToolExecutor()
search()